# Qwen3.8-27B on TPU v5e-8, served over Tailscale

This now **serves the model**, rather than probing. Three gates, in order, and each
one stops rather than degrading:

1. **a TPU must be present** — `/dev/vfio` or the libtpu environment. A CPU
   session is not a TPU session; the measured lesson from the previous run is that
   `machine_shape: TpuV5E8` in the metadata means nothing.
2. **weights must fit** — 27B bf16 is ~55 GB, which fits v5e-8's 128 GB but not
   `/kaggle/working`'s 20 GB, so the cache lives under `/root`.
3. **the server must answer `/health`** before the kernel claims to be serving.

The overlay comes up first so the endpoint is reachable at the node's tailnet
address with no public URL. The `TS_AUTHKEY` secret must be attached **to this
kernel**.

**Unverified:** `vllm-tpu==0.28.0` installation and the XLA cache layout are taken
from the accompanying dataset's manifest, not from a run. The first successful
execution is what will confirm them.


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

def sh(cmd, timeout=900):
    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr)

# ---- 1. TPU inventory ----
# A v5e does not necessarily appear as /dev/accel: it is usually /dev/vfio plus the
# libtpu runtime. Report everything and judge from that, rather than from one guess.
for label, cmd in [
    ("dev accel", "ls -l /dev/accel* 2>&1 | head"),
    ("dev vfio", "ls -l /dev/vfio 2>&1 | head"),
    ("env", "printenv | grep -i -E 'TPU|XRT|PJRT|LIBTPU' | head"),
    ("cpu", "grep -m1 'model name' /proc/cpuinfo; nproc"),
    ("mem", "free -g | head -2"),
    ("disk working", "df -BG /kaggle/working | tail -1"),
    ("disk root", "df -BG /root | tail -1"),
]:
    rc, out = sh(cmd)
    print(f"--- {label} ---")
    print(out.strip()[:400])

# ---- 2. overlay ----
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("TS_AUTHKEY")
except Exception:
    key = os.environ.get("TS_AUTHKEY", "")
tailnet_ip = ""
if key:
    sh("curl -fsSL https://tailscale.com/install.sh -o /tmp/ti.sh && sh /tmp/ti.sh 2>&1 | tail -2")
    SOCK = "/tmp/ts.sock"
    sh(f"nohup tailscaled --tun=userspace-networking --socks5-server=127.0.0.1:1055 "
       f"--socket={SOCK} --state=/kaggle/working/ts.state >/tmp/tailscaled.log 2>&1 & sleep 6")
    p = subprocess.run(["bash", "-lc", f'tailscale --socket={SOCK} up --auth-key="$TSKEY" '
                                     '--hostname=kaggle-tpu --accept-dns=false'],
                       capture_output=True, text=True, env={**os.environ, "TSKEY": key}, timeout=180)
    print("tailscale up rc", p.returncode)
    tailnet_ip = sh(f"tailscale --socket={SOCK} ip -4")[1].strip()
    print("tailnet ipv4:", tailnet_ip)
else:
    print("no TS_AUTHKEY attached to this kernel; the overlay is skipped")

def free_gb_of(mount: str) -> int:
    rc, out = sh(f"df -BG {mount} | tail -1")
    parts = out.split()
    return int(parts[3].rstrip("G")) if len(parts) >= 4 and parts[3].endswith("G") else 0

# Weights belong in the HF cache under /root (the big overlay); /kaggle/working is
# 20 GB and holds outputs only, so judge the mount that will actually hold them.
free_gb = free_gb_of("/root")
work_gb = free_gb_of("/kaggle/working")
print(f"free GB: /root {free_gb}, /kaggle/working {work_gb}")

report = {"free_gb": free_gb, "tailnet_ipv4": tailnet_ip,
          "model": "Qwen/Qwen3.8-27B", "weights_gb_bf16": 55}
Path("/kaggle/working/tpu_report.json").write_text(json.dumps(report, indent=2))

# ---- 3. verdict ----
if free_gb < 70:
    raise SystemExit(
        f"only {free_gb} GB free under /root; Qwen3.8-27B needs ~55 GB of weights plus "
        "compile cache. Nothing was downloaded.")
print(f"disk ok ({free_gb} GB under /root). outputs must stay small: /kaggle/working has {work_gb} GB")


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, timeout=900, env=None):
    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True,
                       timeout=timeout, env=env)
    return r.returncode, (r.stdout + r.stderr)

MODEL = '__MODEL__'
if MODEL.startswith('__'):
    MODEL = 'Qwen/Qwen3.8-27B'
PORT = 8000
MAX_LEN = 8192

# ---- gate 1: a real accelerator ----
rc, vfio = sh("ls /dev/vfio 2>/dev/null | head -3")
rc, envs = sh("printenv | grep -i -E 'TPU|XRT|PJRT|LIBTPU' | head -5")
has_tpu = bool(vfio.strip() or envs.strip())
print("vfio:", vfio.strip()[:120], "| tpu env:", envs.strip()[:120])
if not has_tpu:
    raise SystemExit(
        "no TPU in this session (/dev/vfio absent and no libtpu environment). "
        "Set the accelerator in Settings, confirm the device, and re-run. "
        "Serving a 27B on the CPU fallback is not attempted.")

# ---- gate 2: disk, judged on the mount that will hold the weights ----
def free_gb(mount):
    rc, out = sh(f"df -BG {mount} | tail -1")
    parts = out.split()
    return int(parts[3].rstrip("G")) if len(parts) >= 4 and parts[3].endswith("G") else 0
root_gb, work_gb = free_gb("/root"), free_gb("/kaggle/working")
print(f"disk: /root {root_gb} GB, /kaggle/working {work_gb} GB")
if root_gb < 70:
    raise SystemExit(f"only {root_gb} GB under /root; 27B bf16 needs ~55 GB plus cache")

# HF cache on the big overlay; /kaggle/working holds only small artefacts
os.environ["HF_HOME"] = "/root/.cache/huggingface"

# ---- overlay ----
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("TS_AUTHKEY")
except Exception:
    key = os.environ.get("TS_AUTHKEY", "")
tailnet_ip = ""
if key:
    sh("curl -fsSL https://tailscale.com/install.sh -o /tmp/ti.sh && sh /tmp/ti.sh 2>&1 | tail -1")
    SOCK = "/tmp/ts.sock"
    sh(f"nohup tailscaled --tun=userspace-networking --socks5-server=127.0.0.1:1055 "
       f"--socket={SOCK} --state=/kaggle/working/ts.state >/tmp/tailscaled.log 2>&1 & sleep 6")
    subprocess.run(["bash", "-lc", f'tailscale --socket={SOCK} up --auth-key="$TSKEY" '
                                     '--hostname=kaggle-qwen --accept-dns=false'],
                   capture_output=True, text=True,
                   env={**os.environ, "TSKEY": key}, timeout=180)
    tailnet_ip = sh(f"tailscale --socket={SOCK} ip -4")[1].strip()
    print("tailnet:", tailnet_ip or "(none)")

# ---- install vllm-tpu, as the accompanying dataset's manifest declares ----
VLLM_TPU = "0.28.0"
rc, out = sh(f"pip install -q 'vllm-tpu=={VLLM_TPU}' 2>&1 | tail -5", timeout=1800)
print("install vllm-tpu", VLLM_TPU, "rc", rc)
print(out[-600:])
if rc != 0:
    raise SystemExit(
        f"vllm-tpu=={VLLM_TPU} did not install. The dataset's manifest says that "
        "version with the MTP patch; installing a different one silently is how the "
        "earlier torch_xla attempts failed. Nothing was served.")

# ---- serve ----
cmd = (f"nohup python -m vllm.entrypoints.openai.api_server --model {MODEL} "
       f"--port {PORT} --max-model-len {MAX_LEN} --trust-remote-code "
       f">/tmp/vllm.log 2>&1 &")
print("starting:", cmd)
sh(cmd)

deadline = time.time() + 2400
healthy = False
while time.time() < deadline:
    rc, out = sh(f"curl -sf http://127.0.0.1:{PORT}/health -o /dev/null && echo OK", timeout=30)
    if "OK" in out:
        healthy = True
        break
    time.sleep(30)
    rc, tail = sh("tail -2 /tmp/vllm.log")
    print("waiting:", tail.strip()[-160:])

base = f"http://{tailnet_ip}:{PORT}/v1" if tailnet_ip else f"http://127.0.0.1:{PORT}/v1"
report = {"model": MODEL, "healthy": healthy, "base_url": base,
          "tailnet_ipv4": tailnet_ip, "vllm_tpu": VLLM_TPU}
Path("/kaggle/working/serving.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
if not healthy:
    raise SystemExit("the server never answered /health; see /tmp/vllm.log in the output")
print("SERVING:", base)
